# 5. Multi-Head Attention

**Цель:** Расширить Scaled Dot-Product Attention до многоголового внимания — ключевого компонента трансформеров.

---

In [ ]:
import sys, os, logging, math
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("multihead")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
log.info("Using device: %s", device)

## 5.1 Идея Multi-Head Attention

**Проблема одного внимания:** Одна матрица внимания — один паттерн взаимодействия между токенами.

**Идея:** Использовать h разных "голов" внимания, каждая со своими проекциями Q, K, V:
- Разные головы учат разные типы отношений (синтаксические, семантические, позиционные)
- Каждая голова работает в подпространстве меньшей размерности: d_k = d_model / h
- Результаты конкатенируются и проецируются обратно в d_model

**Формула:**
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W_O$$
$$\text{head}_i = \text{Attention}(QW_Q^i, KW_K^i, VW_V^i)$$

In [ ]:
log.debug("Implementing MultiHeadAttention class")

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention с нуля."""
    
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0, f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Проекционные слои для Q, K, V (общие для всех голов)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        
        # Финальный проекционный слой
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        log.debug("MultiHeadAttention: d_model=%d, n_heads=%d, d_k=%d", d_model, n_heads, self.d_k)
    
    def _split_heads(self, x):
        """(batch, seq_len, d_model) -> (batch, n_heads, seq_len, d_k)"""
        batch, seq_len, _ = x.shape
        x = x.view(batch, seq_len, self.n_heads, self.d_k)
        return x.transpose(1, 2)  # (batch, n_heads, seq_len, d_k)
    
    def _combine_heads(self, x):
        """(batch, n_heads, seq_len, d_k) -> (batch, seq_len, d_model)"""
        batch, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()  # (batch, seq_len, n_heads, d_k)
        return x.view(batch, seq_len, self.d_model)
    
    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q, K, V: (batch, seq_len, d_model)
            mask: (batch, seq_len) — padding mask
        Returns:
            output: (batch, seq_len, d_model)
            attention_weights: (batch, n_heads, seq_len, seq_len)
        """
        batch = Q.size(0)
        
        # 1. Линейные проекции
        Q = self.W_Q(Q)
        K = self.W_K(K)
        V = self.W_V(V)
        log.debug("After projection shapes: Q=%s, K=%s, V=%s", Q.shape, K.shape, V.shape)
        
        # 2. Разделение на головы
        Q = self._split_heads(Q)  # (batch, n_heads, seq_len, d_k)
        K = self._split_heads(K)
        V = self._split_heads(V)
        log.debug("After split: Q=%s, K=%s, V=%s", Q.shape, K.shape, V.shape)
        
        # 3. Scaled Dot-Product Attention для каждой головы
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Маскировка
        if mask is not None:
            # mask: (batch, seq_len) -> (batch, 1, 1, seq_len)
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, V)  # (batch, n_heads, seq_len, d_k)
        log.debug("Per-head output shape: %s", output.shape)
        
        # 4. Объединение голов
        output = self._combine_heads(output)  # (batch, seq_len, d_model)
        
        # 5. Финальная проекция
        output = self.W_O(output)
        log.debug("Final output shape: %s", output.shape)
        
        return output, attention_weights

In [ ]:
log.debug("Testing MultiHeadAttention forward pass")

d_model, n_heads, seq_len, batch = 64, 8, 10, 4
mha = MultiHeadAttention(d_model, n_heads).to(device)

Q = torch.randn(batch, seq_len, d_model, device=device)
K = torch.randn(batch, seq_len, d_model, device=device)
V = torch.randn(batch, seq_len, d_model, device=device)

output, attn_weights = mha(Q, K, V)

print(f"Input shape:        {Q.shape}")
print(f"Output shape:       {output.shape}")
print(f"Attention weights:  {attn_weights.shape}")
print(f"Number of heads:    {n_heads}")
print(f"d_k per head:       {d_model // n_heads}")
print(f"Parameters:         {sum(p.numel() for p in mha.parameters()):,}")
log.info("MultiHeadAttention forward pass OK, params=%d", sum(p.numel() for p in mha.parameters()))

## 5.2 Визуализация: что учат разные головы?

Каждая голова может фокусироваться на разных аспектах взаимодействия токенов.

In [ ]:
log.debug("Visualizing attention patterns across heads")

torch.manual_seed(42)
d_model, n_heads = 32, 4
mha = MultiHeadAttention(d_model, n_heads)

# Используем разные случайные данные, чтобы головы давали разные паттерны
Q = torch.randn(1, 8, d_model) * 2
K = torch.randn(1, 8, d_model) * 2
V = torch.randn(1, 8, d_model) * 2

with torch.no_grad():
    output, attn_weights = mha(Q, K, V)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, ax in enumerate(axes.flat):
    im = ax.imshow(attn_weights[0, i].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {i+1}')
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Attention Patterns Across Heads', fontsize=14)
plt.tight_layout()
plt.show()
log.info("Multi-head attention patterns visualized")

## 5.3 Исследование: влияние числа голов

Проверим, как количество голов влияет на количество параметров и скорость.

In [ ]:
log.debug("Analyzing impact of number of heads")

import time

d_model = 256
head_options = [1, 2, 4, 8, 16, 32]
batch, seq_len = 8, 64

results = []
for n_heads in head_options:
    mha = MultiHeadAttention(d_model, n_heads).to(device)
    params = sum(p.numel() for p in mha.parameters())
    
    Q = torch.randn(batch, seq_len, d_model, device=device)
    K = torch.randn(batch, seq_len, d_model, device=device)
    V = torch.randn(batch, seq_len, d_model, device=device)
    
    # Warmup
    for _ in range(5):
        mha(Q, K, V)
    if device.type == 'mps':
        torch.mps.synchronize()
    
    start = time.perf_counter()
    for _ in range(50):
        mha(Q, K, V)
    if device.type == 'mps':
        torch.mps.synchronize()
    elapsed = (time.perf_counter() - start) / 50
    
    results.append((n_heads, params, elapsed))
    log.info("n_heads=%d, params=%d, time=%.3fms", n_heads, params, elapsed*1000)

n_heads_arr, params_arr, times_arr = zip(*results)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(n_heads_arr, [p/1000 for p in params_arr], 'o-')
axes[0].set_xlabel('Number of heads')
axes[0].set_ylabel('Parameters (thousands)')
axes[0].set_title('Parameters vs Heads')
axes[0].grid(True)

axes[1].plot(n_heads_arr, [t*1000 for t in times_arr], 'o-')
axes[1].set_xlabel('Number of heads')
axes[1].set_ylabel('Time per forward (ms)')
axes[1].set_title('Speed vs Heads')
axes[1].grid(True)

plt.suptitle(f'Impact of Number of Heads (d_model={d_model})')
plt.tight_layout()
plt.show()
log.info("Head impact analysis complete")

## 5.4 Self-Attention: Q=K=V

В энкодере трансформера используется Self-Attention: Q, K, V берутся из одного источника.

In [ ]:
log.debug("Demonstrating self-attention")

d_model, n_heads = 32, 4
mha = MultiHeadAttention(d_model, n_heads)

# Self-attention: один и тот же вход для Q, K, V
x = torch.randn(1, 6, d_model)
output, attn_weights = mha(x, x, x)

print(f"Self-attention input:  {x.shape}")
print(f"Self-attention output: {output.shape}")
print(f"Attention:             {attn_weights.shape}")

# Покажем, что диагональ внимания обычно доминирует (токен "смотрит" на себя)
diag_mean = attn_weights[:, :, range(6), range(6)].mean().item()
print(f"Mean diagonal weight: {diag_mean:.3f} (out of 1.0 per row)")
log.info("Self-attention diagonal dominance: %.3f", diag_mean)

## 5.5 Cross-Attention: Q из одного источника, K, V из другого

В декодере трансформера используется Cross-Attention: Q от декодера, K, V от энкодера.

In [ ]:
log.debug("Demonstrating cross-attention")

d_model, n_heads = 32, 4
mha = MultiHeadAttention(d_model, n_heads)

# Cross-attention: Q из decoder, K, V из encoder
decoder_output = torch.randn(1, 4, d_model)  # 4 токена в декодере
encoder_output = torch.randn(1, 8, d_model)  # 8 токенов в энкодере

output, attn_weights = mha(decoder_output, encoder_output, encoder_output)

print(f"Decoder (Q) shape:   {decoder_output.shape}")
print(f"Encoder (K, V) shape:{encoder_output.shape}")
print(f"Cross-attention output:{output.shape}")
print(f"Attention:             {attn_weights.shape}")

# Визуализация cross-attention
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for i, ax in enumerate(axes.flat):
    im = ax.imshow(attn_weights[0, i].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Cross-Attention Head {i+1}')
    ax.set_xlabel('Encoder position')
    ax.set_ylabel('Decoder position')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Cross-Attention: Decoder attending to Encoder')
plt.tight_layout()
plt.show()
log.info("Cross-attention demonstration complete")

In [ ]:
print("=== Multi-Head Attention complete ===")
print("Topics covered:")
print("  - MultiHeadAttention class from scratch")
print("  - Split/combine heads mechanism")
print("  - Visualization of different head patterns")
print("  - Impact of number of heads on params and speed")
print("  - Self-Attention (Q=K=V) vs Cross-Attention")
print("  - Padding mask for multi-head attention")
log.info("MultiHead attention notebook complete")